# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

***Citation:*** Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will provide an entrypoint for all subsequent exploration.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (Croissant package)
dataset = mlc.Dataset(croissant_url)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

print("\nAvailable attributes in metadata:")
for a in dir(dataset.metadata):
    if not a.startswith('_'):
        print(f"- {a}")


## 2. Data Overview
Examine available record sets, their fields, and their IDs. **Note:** In Croissant, `@id` fields are used to uniquely reference record sets and fields. All exploration and extraction will use these IDs.

In [ ]:
# List all record sets, their IDs, and field IDs

record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
all_rs_ids = []
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    field_ids = []
    for f in rs.fields:
        name = getattr(f, 'name', f.id)
        field_ids.append(f.id)
        print(f"    Field: {name} (@id: {f.id}, type: {f.data_type})")
    all_rs_ids.append(rs.id)
    print()
if not all_rs_ids:
    raise RuntimeError("No record sets found in the dataset's schema.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. Each record set is referenced by its `@id`.

In [ ]:
# Extract all available record sets into DataFrames (by @id)

dataframes = {}
for rs_id in all_rs_ids:
    print(f"Loading records from Record Set: {rs_id}")
    recs = dataset.records(record_set=rs_id)
    rows = list(recs)
    df = pd.DataFrame(rows)
    dataframes[rs_id] = df
    print(f"  Fields: {df.columns.tolist()}")
    print(f"  Shape: {df.shape}\n")
# Choose first record set for demonstration:
example_rs_id = all_rs_ids[0]
print(f"Using record set: {example_rs_id}\nFirst five rows:")
display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic analysis, filtering, and transformation operations to a numeric field from the main record set. For demonstration, we look for numeric fields and use the first one available.

In [ ]:
# Identify a numeric field (float or integer) for EDA
rs = next(rs for rs in dataset.record_sets if rs.id == example_rs_id)
numeric_field_id = None
for f in rs.fields:
    if getattr(f, 'data_type', None) in ["Float", "Integer", "Number"]:
        numeric_field_id = f.id
        print(f"Selected numeric field: {getattr(f, 'name', numeric_field_id)} (@id: {numeric_field_id})")
        break

if not numeric_field_id:
    raise RuntimeError("No numeric field found in chosen record set.")
df = dataframes[example_rs_id]
\n# Convert column to numeric (may have been loaded as string)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records based on threshold
threshold = df[numeric_field_id].mean() # example threshold: mean
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}). Total: {len(filtered_df)}")
display(filtered_df.head())

# Normalize selected field
norm_col = numeric_field_id + "_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nFirst 5 normalized values for {numeric_field_id}:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Optionally group by a string/categorical field if available
group_field_id = None
for f in rs.fields:
    if getattr(f, 'data_type', None) in ["Text", "String"] and f.id != numeric_field_id:
        group_field_id = f.id
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id} (first five groups):")
    display(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship to a group field if one exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=60, ha='right')
    plt.show()

## 6. Conclusion
We have demonstrated how to discover available record sets and fields (using `@id`), load records using `mlcroissant`, inspect and analyze data, filter and normalize numeric fields, group by a category, and visualize distributions. The dataset provides a rich resource for protein mass spectrometry studies in human mast cell extracellular vesicles, and further analysis can focus on identifying biologically relevant patterns in protein abundance and properties.

_For more, visit the [Croissant documentation](https://mlcommons.github.io/croissant/) or the [FAIR² dataset portal](https://sen.science/doi/10.71728/senscience.vq4a-28xa)._